In [1]:
from pathlib import Path
import json
import pandas as pd

CODEQL_RESULTS_DIR = Path("codeql-results")

records = []

for issue_file in CODEQL_RESULTS_DIR.glob("*/*/introduced_issues.json"):

    repo_dir = issue_file.parent.parent.name
    pr_dir = issue_file.parent.name

    owner_repo = repo_dir.split("__", 1)
    if len(owner_repo) == 2:
        owner, repo = owner_repo
    else:
        owner, repo = None, repo_dir

    pr_number = int(pr_dir.replace("pr_", ""))

    with open(issue_file, "r", encoding="utf-8") as f:
        issues = json.load(f)

    # ⚠️ 只保留有 issue 的
    if len(issues) == 0:
        continue

    records.append({
        "owner": owner,
        "repo": repo,
        "pr_number": pr_number,
        "issue_file": str(issue_file),
        "introduced_issue_count": len(issues),
        "issues": issues,
    })

codeql_pr_df = pd.DataFrame(records)

codeql_pr_df = codeql_pr_df.sort_values(
    "introduced_issue_count", ascending=False
).reset_index(drop=True)

codeql_pr_df.head()

,owner,repo,pr_number,issue_file,introduced_issue_count,issues
0,ObservedObserver,async-code,4,codeql-results/ObservedObserver__async-code/pr...,30,"[{'ruleId': 'py/stack-trace-exposure', 'ruleIn..."
1,phospho-app,phosphobot,125,codeql-results/phospho-app__phosphobot/pr_125/...,28,"[{'ruleId': 'py/path-injection', 'ruleIndex': ..."
2,kyryl-opens-ml,no-ocr,16,codeql-results/kyryl-opens-ml__no-ocr/pr_16/in...,8,"[{'ruleId': 'py/path-injection', 'ruleIndex': ..."
3,ChaokunHong,MetaScreener,5,codeql-results/ChaokunHong__MetaScreener/pr_5/...,5,"[{'ruleId': 'py/stack-trace-exposure', 'ruleIn..."
4,NousResearch,atropos,71,codeql-results/NousResearch__atropos/pr_71/int...,5,"[{'ruleId': 'py/stack-trace-exposure', 'ruleIn..."


In [2]:
codeql_pr_df

,owner,repo,pr_number,issue_file,introduced_issue_count,issues
0,ObservedObserver,async-code,4,codeql-results/ObservedObserver__async-code/pr...,30,"[{'ruleId': 'py/stack-trace-exposure', 'ruleIn..."
1,phospho-app,phosphobot,125,codeql-results/phospho-app__phosphobot/pr_125/...,28,"[{'ruleId': 'py/path-injection', 'ruleIndex': ..."
2,kyryl-opens-ml,no-ocr,16,codeql-results/kyryl-opens-ml__no-ocr/pr_16/in...,8,"[{'ruleId': 'py/path-injection', 'ruleIndex': ..."
3,ChaokunHong,MetaScreener,5,codeql-results/ChaokunHong__MetaScreener/pr_5/...,5,"[{'ruleId': 'py/stack-trace-exposure', 'ruleIn..."
4,NousResearch,atropos,71,codeql-results/NousResearch__atropos/pr_71/int...,5,"[{'ruleId': 'py/stack-trace-exposure', 'ruleIn..."
5,ChaokunHong,MetaScreener,4,codeql-results/ChaokunHong__MetaScreener/pr_4/...,4,"[{'ruleId': 'py/url-redirection', 'ruleIndex':..."
6,keephq,keep,5002,codeql-results/keephq__keep/pr_5002/introduced...,4,"[{'ruleId': 'py/log-injection', 'ruleIndex': 4..."
7,AliAkhtari78,SpotifyScraper,48,codeql-results/AliAkhtari78__SpotifyScraper/pr...,3,[{'ruleId': 'py/clear-text-logging-sensitive-d...
8,ObservedObserver,async-code,1,codeql-results/ObservedObserver__async-code/pr...,2,"[{'ruleId': 'py/stack-trace-exposure', 'ruleIn..."
9,bmander,graphserver,37,codeql-results/bmander__graphserver/pr_37/intr...,2,"[{'ruleId': 'py/stack-trace-exposure', 'ruleIn..."


In [3]:
codeql_pr_df['repo'].value_counts()

repo
AGI-Alpha-Agent-v0             9
meta-agent                     6
jaseci                         5
async-code                     3
no-ocr                         2
MetaScreener                   2
SpotifyScraper                 2
zero-finance                   1
HAsmartirrigation              1
just-prompt                    1
mlflow                         1
azure-sdk-tools                1
BentoML                        1
klongpy                        1
skyvern                        1
chai                           1
sentry                         1
DDNS                           1
PotPlayer_ChatGPT_Translate    1
phosphobot                     1
crewAI-tools                   1
haxunit                        1
single-file-agents             1
autogen                        1
litellm                        1
ChaGPT-API-Call                1
graphserver                    1
keep                           1
atropos                        1
pdf-ocr-obsidian               1
Name:

In [5]:
issue_rows = []

for _, row in codeql_pr_df.iterrows():

    for issue in row["issues"]:

        rule_id = issue.get("ruleId")

        message = (issue.get("message") or {}).get("text")

        locations = issue.get("locations") or []

        if locations:
            phys = locations[0].get("physicalLocation") or {}
            artifact = phys.get("artifactLocation") or {}
            region = phys.get("region") or {}

            file_path = artifact.get("uri")
            line = region.get("startLine")
        else:
            file_path = None
            line = None

        # CodeQL没有直接CWE字段，要从rule或tags中拿
        tags = issue.get("properties", {}).get("tags", [])

        cwe = None
        for t in tags:
            if isinstance(t, str) and t.lower().startswith("cwe"):
                cwe = t
                break

        issue_rows.append({
            "owner": row["owner"],
            "repo": row["repo"],
            "pr_number": row["pr_number"],

            "rule_id": rule_id,
            "message": message,

            "file": file_path,
            "line": line,

            "cwe": cwe,
        })

codeql_issue_df = pd.DataFrame(issue_rows)

codeql_issue_df.head()

,owner,repo,pr_number,rule_id,message,file,line,cwe
0,ObservedObserver,async-code,4,py/stack-trace-exposure,[Stack trace information](1) flows to this loc...,server/projects.py,46,None
1,ObservedObserver,async-code,4,py/stack-trace-exposure,[Stack trace information](1) flows to this loc...,server/projects.py,72,None
2,ObservedObserver,async-code,4,py/stack-trace-exposure,[Stack trace information](1) flows to this loc...,server/projects.py,95,None
3,ObservedObserver,async-code,4,py/stack-trace-exposure,[Stack trace information](1) flows to this loc...,server/projects.py,116,None
4,ObservedObserver,async-code,4,py/stack-trace-exposure,[Stack trace information](1) flows to this loc...,server/projects.py,138,None


In [7]:
codeql_issue_df['cwe'].value_counts()

Series([], Name: count, dtype: int64)